In [ ]:
import sys
import subprocess

required = ["sentence-transformers", "datasets", "scipy", "pandas", "scikit-learn", "psutil"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])


In [ ]:
import os
import random
import time
import numpy as np
import pandas as pd
import psutil
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/paraphrase-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
subset_size = 400
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 256 if device == "mps" else 64
process = psutil.Process(os.getpid())
start_time = time.time()
start_rss_mb = process.memory_info().rss / (1024 ** 2)

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "subset_size": subset_size,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
    "start_rss_mb": round(start_rss_mb, 2),
})


In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
df = df.head(min(subset_size, len(df))).copy()

print({
    "num_examples_after_cap": len(df),
    "columns": df.columns.tolist(),
    "label_min": float(df["label"].min()),
    "label_max": float(df["label"].max()),
    "label_mean": float(df["label"].mean()),
})
print(df.head())


In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()

load_rss_mb = process.memory_info().rss / (1024 ** 2)
print({
    "loaded_model": model_name,
    "device": device,
    "rss_mb_after_model_load": round(load_rss_mb, 2),
})


In [ ]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

encode_start = time.time()

emb1 = model.encode(
    sentences1,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=False,
    show_progress_bar=False,
)

emb2 = model.encode(
    sentences2,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=False,
    show_progress_bar=False,
)

encode_seconds = time.time() - encode_start
embedding_dim = int(emb1.shape[1])

dot_product_raw = np.sum(emb1 * emb2, axis=1)
dot_min = float(dot_product_raw.min())
dot_max = float(dot_product_raw.max())
if dot_max > dot_min:
    predicted_score_0_5 = 5.0 * (dot_product_raw - dot_min) / (dot_max - dot_min)
else:
    predicted_score_0_5 = np.full_like(dot_product_raw, 2.5, dtype=np.float32)

emb1_mb = emb1.nbytes / (1024 ** 2)
emb2_mb = emb2.nbytes / (1024 ** 2)
post_encode_rss_mb = process.memory_info().rss / (1024 ** 2)

print({
    "embedding_dim": embedding_dim,
    "emb1_shape": tuple(emb1.shape),
    "emb2_shape": tuple(emb2.shape),
    "emb1_mb": round(emb1_mb, 2),
    "emb2_mb": round(emb2_mb, 2),
    "dot_min": round(dot_min, 6),
    "dot_max": round(dot_max, 6),
    "encode_seconds": round(encode_seconds, 3),
    "rss_mb_after_encode": round(post_encode_rss_mb, 2),
})


In [ ]:
pearson_dot_raw = pearsonr(dot_product_raw, labels).statistic
spearman_dot_raw = spearmanr(dot_product_raw, labels).statistic
pearson_dot_rescaled = pearsonr(predicted_score_0_5, labels).statistic
spearman_dot_rescaled = spearmanr(predicted_score_0_5, labels).statistic

results_df = df.copy()
results_df["dot_product_raw"] = dot_product_raw
results_df["predicted_score_0_5"] = predicted_score_0_5

print(results_df[["sentence1", "sentence2", "label", "dot_product_raw", "predicted_score_0_5"]].head(10))


In [ ]:
runtime_seconds = time.time() - start_time
final_rss_mb = process.memory_info().rss / (1024 ** 2)

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"subset_size_used: {len(df)}")
print(f"embedding_dimensionality: {embedding_dim}")
print(f"pearson_dot_raw: {pearson_dot_raw:.6f}")
print(f"spearman_dot_raw: {spearman_dot_raw:.6f}")
print(f"pearson_dot_rescaled_0_5: {pearson_dot_rescaled:.6f}")
print(f"spearman_dot_rescaled_0_5: {spearman_dot_rescaled:.6f}")
print(f"encode_seconds: {encode_seconds:.3f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
print(f"rss_mb_start: {start_rss_mb:.2f}")
print(f"rss_mb_after_model_load: {load_rss_mb:.2f}")
print(f"rss_mb_after_encode: {post_encode_rss_mb:.2f}")
print(f"rss_mb_final: {final_rss_mb:.2f}")
print(f"embedding_arrays_total_mb: {(emb1_mb + emb2_mb):.2f}")
